# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [118]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

RISK_BUDGET_PCT = 5.0
ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
namespace={'__name__':'hyperliquid_multi_account_analytics'}
implementation_paths=[
    Path('hyperliquid_account_analytics_implementation.ipynb'),
    Path('hyperliquid_multi_account_analytics_implementation.ipynb'),
]
for implementation_path in implementation_paths:
    implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
    for cell in implementation['cells']:
        if cell['cell_type']=='code':
            source=''.join(cell['source'])
            if source.strip():
                exec(compile(source,f'{implementation_path} cell','exec'),namespace)
MultiWalletAnalytics=namespace['MultiWalletAnalytics']
analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_risk_budget_pct=RISK_BUDGET_PCT
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [119]:
%%capture
_ = analytics.fetch_data()


## Account / PnL

Combined account and risk summary. Daily PnL and calendar-period cutoffs use US Eastern time.


In [120]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$2,918.78"
Portfolio value,"$3,004.20 (combined live wallet value)"
Account 1D (ET),2.39%
Realized today (ET),$-1.58
Realized today (% of account value),-0.05%
Unrealized PnL,$-155.51
Unrealized PnL (% of account value),-5.33%
Unrealized 1D change (ET),n/a
Average realized PnL / day,$-0.99
Average unrealized PnL change / day,n/a


## Account charts

Combined account value and PnL charts.


In [121]:
analytics.show_account_charts()


## Account vs BTC

Combined account comparison against BTC.


In [122]:
analytics.show_account_vs_btc()


Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [123]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by fixed-fractional sizing grouped by asset and side and an aggregate summary. Set `RISK_BUDGET_PCT` (5% per asset), `ATR_PERIOD` (14 completed daily candles), and `ATR_STOP_MULTIPLE` (1.25) in the setup cell. Stops are placed 1.25 ATR from the current mark, and take-profit is set the same distance in the favorable direction (1:1 risk/reward). Same-side positions across wallets are combined; long and short rows remain separate. If both sides are open for one asset, they share that asset's risk budget equally.


In [124]:
analytics.show_open_positions()


Wallet,Coin,Side,Liq %,ROE,uPnL,Funding,Funding (1h),Funding APR,Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Liq Px,Lev.,Sz
c248,IMX,SHORT,65.96%,1.77%,$3.51,$0.23,0.001250%,10.95%,0.12332,0.12289,$988,$198,Cross,65.84%,-$0.14,0.20395,5.00x,"-8,040.50"
07f2,ADA,SHORT,98.00%,2.87%,$6.27,-$0.21,-0.000923%,-8.09%,0.193547,0.19216,$869,$217,Cross,20.46%,-$0.18,0.38048,4.00x,"-4,522.00"
07f2,PENGU,SHORT,129.02%,2.50%,$3.16,$0.04,0.001250%,10.95%,0.007292,0.007256,$630,$126,Cross,11.87%,-$0.09,0.016617,5.00x,"-86,832.00"
07f2,EIGEN,SHORT,137.79%,2.08%,$3.09,$0.10,0.002485%,21.77%,0.1907,0.1898,$590,$147,Cross,13.89%,-$0.22,0.45132,4.00x,"-3,108.22"
07f2,xyz:JIO,LONG,7.18%,-9.75%,-$5.79,$24.29,0.000183%,1.60%,14.22973,14.091,$588,$69,Isolated,6.54%,-$0.08,13.08,8.46x,41.70
07f2,xyz:GOOGL,LONG,n/a,-52.03%,-$100.96,-$2.28,0.001047%,9.17%,343.02,283.53,$481,$160,Cross,15.11%,-$0.05,0,3.00x,1.70
07f2,io:PRL,SHORT,15.96%,2.25%,$2.43,$2.96,0.022366%,195.93%,1.48825,1.4771,$321,$113,Isolated,10.68%,-$0.03,1.7129,2.83x,-217.60
c248,INIT,SHORT,19.23%,12.44%,$11.21,-$0.03,0.001250%,10.95%,0.18359,0.17594,$259,$101,Isolated,33.74%,-$0.04,0.20978,2.56x,"-1,473.00"
07f2,xyz:RDDT,SHORT,4.00%,-9.13%,-$1.76,$0.30,0.005232%,45.83%,92.041,92.881,$194,$18,Isolated,1.68%,-$0.02,96.596,10.87x,-2.09
07f2,xyz:AAPL,LONG,n/a,-1.76%,-$0.91,-$0.02,0.000625%,5.48%,346.87,344.84,$155,$52,Cross,4.87%,-$0.01,0,3.00x,0.45


Fixed-fractional sizing: risk budget 5.00% of combined account value per asset; stop distance 1.25 x daily ATR(14). Opposing sides, when both open, split the asset budget equally.
Limited daily history; ATR uses all completed candles available (shorter than 14 days) for: xyz:JIO.
ATR stop-distance colors: green <2%, orange 2-5%, red >5%; red mark means the position is out of the money; stop prices are highlighted red.


Asset,Side,ATR (%),ATR Period Used,ATR Stop Distance (%),Avg Entry,Mark,Stop,Take Profit,Open - Target (coins),Open - Target (%),Open Risk ($)
ADA,SHORT,6.61%,14 days,8.27%,0.193547,0.19216,0.20805,0.17627,"4,931.73",-52.17%,$71.85
AERO,LONG,10.12%,14 days,12.64%,0.832395,0.64012,0.55918,0.72106,"-1,736.82",-93.59%,$9.63
EIGEN,SHORT,10.76%,14 days,13.45%,0.1907,0.1898,0.21533,0.16427,"2,775.11",-47.17%,$79.36
IMX,SHORT,9.13%,14 days,11.42%,0.12332,0.12289,0.13692,0.10886,"2,664.41",-24.89%,$112.82
INIT,SHORT,15.01%,14 days,18.76%,0.18359,0.17594,0.20895,0.14293,"3,076.97",-67.63%,$48.63
PENGU,SHORT,8.46%,14 days,10.58%,0.007292,0.007256,0.0080238,0.0064882,"108,816",-55.62%,$66.67
PROVE,SHORT,6.52%,14 days,8.15%,0.203334,0.20819,0.22516,0.19122,"8,309.89",-93.87%,$9.21
RSR,SHORT,10.85%,14 days,13.56%,0.001385,0.001573,0.0017864,0.0013596,"609,237",-86.54%,$20.22
ZRO,SHORT,11.47%,14 days,14.34%,1.65869,1.8577,2.1241,1.5913,499.922,-88.67%,$17.02
io:PRL,SHORT,21.20%,14 days,26.50%,1.48825,1.4771,1.8685,1.0857,166.146,-43.30%,$85.18


Metric,Value
Wallets,2
Position value,"$5,637"
uPnL,$-155.51
ROE,-11.38%
Margin,"$1,367"
Margin % bal,43.26%
Leverage (Pos / Margin),4.12x
Funding,$25.70
Account value,"$3,004.20"


ROE is unrealized PnL divided by used margin; a negative ROE means the open positions are currently at an unrealized loss relative to the margin supporting them.


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [125]:
analytics.show_realized_pnl()


=== REALIZED SPOT ===


Metric,Value
Realized PnL,$22.35 (0.74%)
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$22.35 (0.74%)
QTD,$22.35 (0.74%)
YTD,$22.35 (0.74%)
W,3
Win %,100.00%
Avg W,$7.45
L,0


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-77.81 (-2.59%)
24H,$9.09 (0.30%)
7D,$339.68 (11.31%)
30D,$446.21 (14.85%)
QTD,$446.12 (14.85%)
YTD,$445.99 (14.85%)
W,181
Win %,70.16%
Avg W,$8.62
L,77



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$83.55,107.39%
Fees,$-105.47,-135.55%
Funding,$-55.89,-71.83%
Net perp PnL,$-77.81,


Liquidations: 0 of 258 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [126]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$247,643.94",$-21.92,-0.01%,$-0.09,57,0.102,0.284
Spot,"$1,397.53",$22.35,1.60%,$16.00,5,-0.509,-0.462


## PnL attribution

Combined realized and open-position PnL by asset.


In [127]:
analytics.show_pnl_attribution()


=== CONSOLIDATED PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,BTC,11,$-474.42,-18.98%,$0.00,0.00%,$-474.42,-17.67%
Perp,kBONK,3,$-397.06,-15.88%,$0.00,0.00%,$-397.06,-14.79%
Perp,ZORA,21,$304.49,12.18%,$0.00,0.00%,$304.49,11.34%
Perp,KAITO,4,$-193.84,-7.75%,$0.00,0.00%,$-193.84,-7.22%
Perp,HYPE,11,$146.54,5.86%,$0.00,0.00%,$146.54,5.46%
Perp,ETH,14,$116.62,4.67%,$0.00,0.00%,$116.62,4.34%
Perp,VVV,40,$116.26,4.65%,$0.00,0.00%,$116.26,4.33%
Perp,xyz:GOOGL,0,$-2.61,-0.10%,$-100.96,-47.04%,$-103.56,-3.86%
Perp,io:ANTH,2,$-86.69,-3.47%,$0.00,0.00%,$-86.69,-3.23%
Perp,AAVE,4,$54.17,2.17%,$0.00,0.00%,$54.17,2.02%


Net attributed PnL across wallets: $-210.72.
